In [1]:
import pandas as pd
import scipy.stats as stats
import altair as alt
import numpy as np
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score
from scipy.stats import fisher_exact
from statsmodels.stats.proportion import proportions_ztest

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [5]:
sge_set=pd.read_excel('./Data/sge_data_for_qc/spliceai_benchmarking/20260512_SGESplicingSet.xlsx')
curated_set=pd.read_excel('./Data/sge_data_for_qc/spliceai_benchmarking/20260512_CuratedSplicingSet.xlsx')
clinvar_set = pd.read_excel('./Data/sge_data_for_qc/spliceai_benchmarking/20260512_ClinVarSplicingSet.xlsx')

In [6]:
import itertools

# Maps SGE simplified_consequence labels → curated/ClinVar VEP SO term labels
CONSEQUENCE_MAP = {
    'Intron':           ['intron_variant'],
    'Splice Region':    ['splice_donor_region_variant', 'splice_polypyrimidine_tract_variant',
                         'splice_donor_5th_base_variant', 'splice_region_variant'],
    'Canonical Splice': ['splice_acceptor_variant', 'splice_donor_variant'],
    'Missense':         ['missense_variant'],
    'Synonymous':       ['synonymous_variant'],
}

SGE_LABEL_COL    = 'auth_reported_func_class'
SGE_POS_LABEL    = 'functionally_abnormal'
SGE_NEG_LABEL    = 'functionally_normal'

CURATED_LABEL_COL = 'splice_consequence'
CURATED_POS_LABEL = 'abnormal'
CURATED_NEG_LABEL = 'normal'

CLINVAR_LABEL_COL = 'ClinicalSignificance'
CLINVAR_POS_LABEL = 'PLP'
CLINVAR_NEG_LABEL = 'BLB'
CLINVAR_CONS_COL  = 'first_consequence'


def build_confusion_matrix(df, label_col, pos_label, neg_label,
                            score_col='maxSpliceAI', threshold=0.2):
    """Returns [[TN, FP], [FN, TP]] at a fixed threshold."""
    sub = df[df[label_col].isin([pos_label, neg_label])].dropna(subset=[score_col])
    pos = sub[sub[label_col] == pos_label]
    neg = sub[sub[label_col] == neg_label]
    tp = int((pos[score_col] >= threshold).sum())
    fn = int((pos[score_col] <  threshold).sum())
    tn = int((neg[score_col] <  threshold).sum())
    fp = int((neg[score_col] >= threshold).sum())
    return np.array([[tn, fp], [fn, tp]])


def compare_components(cms, labels):
    """
    Compare sensitivity and specificity across N datasets.
    cms:    list of [[TN, FP], [FN, TP]] confusion matrices
    labels: list of dataset names, same length as cms

    Bonferroni correction is applied across all pairwise tests for both
    metrics simultaneously: factor = n_pairs * 2.
    Returns a dict keyed by metric with per-dataset values, n counts,
    and corrected p-values for every pair as 'p_{A}_vs_{B}'.
    """
    n_pairs = len(cms) * (len(cms) - 1) // 2
    bonferroni_factor = n_pairs * 2

    metric_counts = {
        'sensitivity': lambda cm: (cm[1, 1], cm[1, 0]),  # TP, FN
        'specificity': lambda cm: (cm[0, 0], cm[0, 1]),  # TN, FP
    }

    results = {}
    for metric, get_counts in metric_counts.items():
        entry = {}
        for label, cm in zip(labels, cms):
            s, f = get_counts(cm)
            entry[label] = s / (s + f)
            entry[f'n_{label}'] = int(s + f)

        for la, lb in itertools.combinations(labels, 2):
            s_a, f_a = get_counts(cms[labels.index(la)])
            s_b, f_b = get_counts(cms[labels.index(lb)])
            _, p = proportions_ztest([s_a, s_b], [s_a + f_a, s_b + f_b])
            entry[f'p_{la}_vs_{lb}'] = min(p * bonferroni_factor, 1.0)

        results[metric] = entry

    return results

In [7]:
RNA_FILTERABLE = {'Splice Region', 'Missense', 'Synonymous'}

def get_sge_sub(sge_set, group_name, rna_filtered=False):
    sub = sge_set[sge_set['simplified_consequence'] == group_name]
    if rna_filtered:
        # Restrict to variants with RNA data, then keep only RNA-confirmed LoF as positives
        sub = sub[sub['rna_consequence'].notna()]
        sub = sub[
            (sub[SGE_LABEL_COL] == SGE_NEG_LABEL) |
            ((sub[SGE_LABEL_COL] == SGE_POS_LABEL) & (sub['rna_consequence'] == 'low'))
        ]
    return sub


rows = []
for group_name, curated_consequences in CONSEQUENCE_MAP.items():
    cur_sub = curated_set[curated_set['simplified_consequence'].isin(curated_consequences)]
    clv_sub = clinvar_set[clinvar_set[CLINVAR_CONS_COL].isin(curated_consequences)]

    variants = [(group_name, False)]
    if group_name in RNA_FILTERABLE:
        variants.append((f'{group_name} (Low RNA)', True))

    for label, rna_filtered in variants:
        sge_sub = get_sge_sub(sge_set, group_name, rna_filtered)

        sge_pos = sge_sub[SGE_LABEL_COL].eq(SGE_POS_LABEL).sum()
        sge_neg = sge_sub[SGE_LABEL_COL].eq(SGE_NEG_LABEL).sum()
        cur_pos = cur_sub[CURATED_LABEL_COL].eq(CURATED_POS_LABEL).sum()
        cur_neg = cur_sub[CURATED_LABEL_COL].eq(CURATED_NEG_LABEL).sum()
        clv_pos = clv_sub[CLINVAR_LABEL_COL].eq(CLINVAR_POS_LABEL).sum()
        clv_neg = clv_sub[CLINVAR_LABEL_COL].eq(CLINVAR_NEG_LABEL).sum()

        if min(sge_pos, sge_neg, cur_pos, cur_neg, clv_pos, clv_neg) < 5:
            print(f'Skipping {label}: insufficient class counts '
                  f'(SGE +{sge_pos}/-{sge_neg}, Curated +{cur_pos}/-{cur_neg}, '
                  f'ClinVar +{clv_pos}/-{clv_neg})')
            continue

        cm_sge = build_confusion_matrix(sge_sub, SGE_LABEL_COL, SGE_POS_LABEL, SGE_NEG_LABEL)
        cm_cur = build_confusion_matrix(cur_sub, CURATED_LABEL_COL, CURATED_POS_LABEL, CURATED_NEG_LABEL)
        cm_clv = build_confusion_matrix(clv_sub, CLINVAR_LABEL_COL, CLINVAR_POS_LABEL, CLINVAR_NEG_LABEL)

        result = compare_components(
            [cm_sge, cm_cur, cm_clv],
            ['SGE', 'Curated', 'ClinVar']
        )

        for metric, vals in result.items():
            rows.append({
                'consequence':          label,
                'metric':               metric,
                'SGE':                  vals['SGE'],
                'Curated':              vals['Curated'],
                'ClinVar':              vals['ClinVar'],
                'p_SGE_vs_Curated':     vals['p_SGE_vs_Curated'],
                'p_SGE_vs_ClinVar':     vals['p_SGE_vs_ClinVar'],
                'p_Curated_vs_ClinVar': vals['p_Curated_vs_ClinVar'],
                'n_SGE':                vals['n_SGE'],
                'n_Curated':            vals['n_Curated'],
                'n_ClinVar':            vals['n_ClinVar'],
            })

comparison_df = pd.DataFrame(rows)

fmt = lambda x: f'{x:.3f}'
sensitivity_df = comparison_df[comparison_df['metric'] == 'sensitivity'].drop(columns='metric').reset_index(drop=True)
specificity_df = comparison_df[comparison_df['metric'] == 'specificity'].drop(columns='metric').reset_index(drop=True)

print('=== Sensitivity ===')
print(sensitivity_df.to_string(index=False, float_format=fmt))
print()
print('=== Specificity ===')
print(specificity_df.to_string(index=False, float_format=fmt))

Skipping Intron: insufficient class counts (SGE +343/-8403, Curated +4/-3, ClinVar +29/-33320)
Skipping Canonical Splice: insufficient class counts (SGE +1033/-102, Curated +92/-1, ClinVar +2914/-31)
Skipping Missense: insufficient class counts (SGE +4831/-25819, Curated +92/-139, ClinVar +2/-1)
Skipping Missense (Low RNA): insufficient class counts (SGE +137/-9016, Curated +92/-139, ClinVar +2/-1)
=== Sensitivity ===
            consequence   SGE  Curated  ClinVar  p_SGE_vs_Curated  p_SGE_vs_ClinVar  p_Curated_vs_ClinVar  n_SGE  n_Curated  n_ClinVar
          Splice Region 0.747    0.873    0.917             0.106             0.000                 1.000    747         71        337
Splice Region (Low RNA) 0.840    0.873    0.917             1.000             1.000                 1.000     25         71        337
             Synonymous 0.329    0.500    0.348             1.000             1.000                 1.000    161         14         23
   Synonymous (Low RNA) 0.529    0.500